In [11]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate

In [12]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# Generates random bits using quantum measurements of superpositions
def get_quantum_random_bits(num_bits):
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits))
    qc.measure(range(num_bits), range(num_bits))
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1, memory=True)
    return [int(b) for b in job.result().get_memory()[0][::-1]]

In [13]:
# Formats and prints the results including Eve's interference
def print_bb84_attacker_table(alice_bits, alice_bases, eve_bases, eve_results, bob_bases, bob_results):
    basis_sym = {0: '+ (Std)', 1: 'x (Diag)'}
    print("=" * 105)
    print(f"{'Qubit':<6} | {'Alice Bit':<9} | {'Alice Base':<10} | {'Eve Base':<10} | {'Eve Bit':<8} | {'Bob Base':<10} | {'Bob Bit':<8} | {'Match? (A vs B)'}")
    print("-" * 105)

    for i in range(len(alice_bits)):
        # Only evaluate matches where Alice and Bob used the same basis
        if alice_bases[i] == bob_bases[i]:
            if alice_bits[i] == bob_results[i]:
                match = "Match"
            else:
                match = "*** ERROR ***" # Eve's interference caused a discrepancy!
        else:
            match = "Discarded"

        print(f"{i:<6} | {alice_bits[i]:<9} | {basis_sym[alice_bases[i]]:<10} | "
              f"{basis_sym[eve_bases[i]]:<10} | {eve_results[i]:<8} | "
              f"{basis_sym[bob_bases[i]]:<10} | {bob_results[i]:<8} | {match}")
    print("=" * 105)

In [14]:
def sim_bb84_attacker(num_bits=20):
    print(f"--- Starting BB84 Attack Simulation with {num_bits} qubits ---\n")

    # Alice's Preparation
    alice_bits = get_quantum_random_bits(num_bits)
    alice_bases = get_quantum_random_bits(num_bits)

    # We need 2 classical registers:
    # First `num_bits` for Eve's measurements, second `num_bits` for Bob's.
    qc = QuantumCircuit(num_bits, 2 * num_bits)

    for i in range(num_bits):
        if alice_bits[i] == 1: qc.x(i)
        if alice_bases[i] == 1: qc.h(i)

    # Eve intercepts the qubits before they reach Bob. She must guess bases too.
    eve_bases = get_quantum_random_bits(num_bits)

    for i in range(num_bits):
        if eve_bases[i] == 1: qc.h(i)

        # Eve measures the qubit (collapsing the state) into classical bits 0 to num_bits-1
        qc.measure(i, i)

        # After measuring, Eve tries to cover her tracks by resending the
        # state she just measured to Bob.
        if eve_bases[i] == 1: qc.h(i)

    # Bob's Measurement
    bob_bases = get_quantum_random_bits(num_bits)

    for i in range(num_bits):
        if bob_bases[i] == 1: qc.h(i)
        # Bob measures the (now tampered) qubit into classical bits num_bits to 2*num_bits-1
        qc.measure(i, i + num_bits)

    # Execute simulation
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1, memory=True)

    # Process Classical Register Results
    full_results = job.result().get_memory()[0][::-1]
    eve_results = [int(b) for b in full_results[0:num_bits]]
    bob_results = [int(b) for b in full_results[num_bits:2*num_bits]]

    # Print step-by-step visual tracker
    print_bb84_attacker_table(alice_bits, alice_bases, eve_bases, eve_results, bob_bases, bob_results)

    # Sifting and threat detection
    sifted_alice = []
    sifted_bob = []

    for i in range(num_bits):
        if alice_bases[i] == bob_bases[i]:
            sifted_alice.append(alice_bits[i])
            sifted_bob.append(bob_results[i])

    # To detect Eve, Alice and Bob publicly share a portion of their sifted key.
    # Usually they sacrifice half of the key to test for errors.
    test_size = len(sifted_alice) // 2

    test_alice = sifted_alice[:test_size]
    test_bob = sifted_bob[:test_size]

    errors = sum(1 for a, b in zip(test_alice, test_bob) if a != b)
    error_rate = errors / test_size if test_size > 0 else 0

    print(f"\n--- SECURITY ANALYSIS ---")
    print(f"Total Sifted Bits: {len(sifted_alice)}")
    print(f"Bits Sacrificed for Error Checking: {test_size}")
    print(f"Errors Found in Sample: {errors}")
    print(f"Error Rate: {error_rate:.2%}")

    # Threshold check: Theory states Eve causes ~25% error rate on average.
    # We use a threshold > 0% here, or a standard 10%-15% tolerance in noisy real systems.
    if error_rate > 0.10:
        print("\nALERT: EAVESDROPPER DETECTED! (Error threshold breached) Protocol aborted.")
    else:
        print("\nChannel secure. Remaining bits can be used for the final key.")

In [15]:
sim_bb84_attacker(20)

--- Starting BB84 Attack Simulation with 20 qubits ---

Qubit  | Alice Bit | Alice Base | Eve Base   | Eve Bit  | Bob Base   | Bob Bit  | Match? (A vs B)
---------------------------------------------------------------------------------------------------------
0      | 0         | + (Std)    | x (Diag)   | 1        | + (Std)    | 0        | Match
1      | 0         | + (Std)    | + (Std)    | 0        | x (Diag)   | 0        | Discarded
2      | 0         | + (Std)    | + (Std)    | 0        | + (Std)    | 0        | Match
3      | 0         | x (Diag)   | + (Std)    | 1        | + (Std)    | 1        | Discarded
4      | 0         | + (Std)    | + (Std)    | 0        | x (Diag)   | 1        | Discarded
5      | 0         | + (Std)    | + (Std)    | 0        | + (Std)    | 0        | Match
6      | 0         | x (Diag)   | x (Diag)   | 0        | x (Diag)   | 0        | Match
7      | 0         | x (Diag)   | x (Diag)   | 0        | + (Std)    | 1        | Discarded
8      | 1         |

## Output Explanation
This simulation demonstrates the Intercept-Resend attack, where Eve attempts to steal the key by sitting between Alice and Bob. Because of the laws of quantum mechanics, Eve cannot "peek" at a qubit without potentially changing its state.

**Why do Errors Occur? (with Qubit 8 as example)**\
After Alice encodes a `1` with the diagonal `x` basis, Eve intercepts it but guesses the standard `+` basis. This forces the qubit to "collapse" into the standard basis.

Eve happens to measure a `1`, and she sends a Standard `1` to Bob.
Bob chooses the diagonal basis `x`. Since Alice also chose diagonal, this should have been a matching bit for their key.

**The Result:** Because Eve changed the qubit to standard, Bob's diagonal measurement becomes random and measures a `0` while Alice has a `1`. Comparing both of their bases publicly would signal that an intruder has tampered with the qubit.

## Security Analysis
To ensure the channel is safe, the protocol requires Alice and Bob to sacrifice a portion of their potential key to check for errors.
- 6 bits from their sifted key to compare bit-values openly.
- 2 errors found in the small sample.

This results in a 33.33% error rate.